# 02 — Engenharia de features

Duas passadas:

**Passada 1** (por ativo): retornos/lags → vol. realizada → Garman-Klass → GARCH(1,1)
(refit periódico dos parâmetros + atualização diária da variância condicional com
retornos observados, sem lookahead) → skew/curtose móveis → spread de
Corwin-Schultz → vol. implícita (VIX + Ibov/Bloomberg). Salva
`data/processed/features.parquet` — **repositório bruto de todas as features**,
sem filtro (a seleção final é por split, ver Passada 2).

**Passada 2** (painel *pooled*, todos os ativos juntos): estacionariedade
(ADF/KPSS, **por ativo separadamente** — rodar num painel concatenado
cross-asset quebraria a ordenação temporal que o teste assume) → correlação
→ filtro VIF (>10), estes dois últimos no painel pooled. **Isto é um
diagnóstico exploratório na amostra completa, não a seleção final de
features** — a seleção de verdade entra dentro de cada split walk-forward
(Semana 6), para não vazar informação de teste para a escolha de quais
features usar.

Painel alinhado ao **calendário real de pregão da B3** (não dias-úteis
genéricos) — dias em que a B3 está fechada não entram no painel; dias em que
a bolsa de origem de um ativo estrangeiro (CBOT para soja, ICE para café)
está fechada mas a B3 está aberta contam como "mercado fechado daquele
ativo", tratados pela mesma regra de gap (ver `align_and_fill_ohlc`).

Soja e café (genérico contínuo do Bloomberg, já ajustado por rolagem — ver
célula de política de futuros abaixo) são **convertidos de USD para
BRL-equivalente** usando o câmbio do próprio dia antes de calcular qualquer
feature — a carteira é medida na perspectiva de um investidor brasileiro,
então retorno/vol/GARCH desses ativos precisam incorporar o risco cambial,
não só o risco de preço da commodity em dólar.

`NaN` estrutural (burn-in do GARCH, `implied_vol_br` pré-2015, gaps de dados
> 3 dias de pregão) é mantido em `features.parquet` — o corte final da
amostra de modelagem fica para quando o dataset de treino for montado
(Semana 5-6).

**Política de futuros:** soja/café vêm do genérico contínuo do Bloomberg
(`data/raw/futures_bloomberg_soja.xlsx`, `data/raw/futures_bloomberg_cafe.xlsx`),
já ajustado por rolagem por razão/percentual (não por diferença — evita preço
negativo e preserva log-retorno). Não aplicamos `adjust_ohlc_roll` em cima —
o arquivo já vem ajustado. `adjust_futures_roll`/`adjust_ohlc_roll` continuam
disponíveis em `src/data/collectors.py` para o dia em que tivermos fatores
verificados por contrato (Yahoo não retém histórico de contratos individuais
expirados — não dá para reconstruir esses fatores a partir dele).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.features import align_and_fill_ohlc, build_asset_features, convert_ohlc_to_brl, get_b3_trading_days, equal_weighted_portfolio_return
from src.data.feature_selection import correlation_matrix, stationarity_report, vif_filter
from src.data.collectors import load_bloomberg_futures_adjusted

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
end = config["sample"]["end_date"]

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Montar o painel: início comum entre os 7 ativos e tratamento de missing

Soja/café usam o genérico contínuo do Bloomberg (já ajustado por rolagem — ver
célula de política de futuros acima; CEPEA fica só para robustez futura). O
início comum do painel é definido pela equity/FX mais tardia (as commodities
cobrem 2005 inteiro, então nunca são elas que limitam).

In [ ]:
equities_fx = pd.read_parquet(RAW_DIR / "equities_fx.parquet")
implied_vol = pd.read_parquet(RAW_DIR / "implied_vol.parquet")

BLOOMBERG_FUTURES_FILES = {
    "ZS=F": RAW_DIR / "futures_bloomberg_soja.xlsx",
    "KC=F": RAW_DIR / "futures_bloomberg_cafe.xlsx",
}

equity_fx_tickers = [a["ticker"] for a in config["assets"] if a["category"] in ("equity", "fx")]
futures_tickers = [a["ticker"] for a in config["assets"] if a["category"] == "commodity"]

core_ohlc = {t: equities_fx[t] for t in equity_fx_tickers}
core_ohlc.update({t: load_bloomberg_futures_adjusted(BLOOMBERG_FUTURES_FILES[t]) for t in futures_tickers})

common_start = max(df["Close"].dropna().index.min() for df in core_ohlc.values())
print("início comum do painel:", common_start.date())

In [ ]:
aligned_native = {ticker: align_and_fill_ohlc(df, common_start, end) for ticker, df in core_ohlc.items()}

# soja/café (USD) -> BRL-equivalente, usando o câmbio do próprio dia já alinhado/tratado
fx_close_brl_per_usd = aligned_native["BRL=X"]["Close"]
aligned_ohlc = dict(aligned_native)
for ticker in futures_tickers:
    aligned_ohlc[ticker] = convert_ohlc_to_brl(aligned_native[ticker], fx_close_brl_per_usd)

# Mesma regra dos preços: o VIX não é cotado em feriados americanos com a B3 aberta.
# Sem o preenchimento causal (até 3 pregões), cada buraco isolado derrubaria todas
# as janelas de lookback da LSTM que passam por ele.
b3_days = get_b3_trading_days(common_start, end)
implied_vol_aligned = implied_vol.reindex(b3_days).ffill(limit=3)

## Passada 1 — features por ativo

In [4]:
features_by_asset = {}
for ticker, ohlc in aligned_ohlc.items():
    asset_features = build_asset_features(
        ohlc,
        lags=tuple(config["features"]["returns_lags"]),
        windows=tuple(config["features"]["rolling_windows"]),
    )
    if ticker in [a["ticker"] for a in config["assets"] if a["category"] == "fx"]:
        asset_features = asset_features.drop(columns="volume_normalized")
    features_by_asset[ticker] = pd.concat([asset_features, implied_vol_aligned], axis=1)

features_by_asset["SLCE3.SA"].tail()

,return_1d,return_5d,return_21d,realized_vol_21d,realized_vol_63d,garman_klass_vol_21d,garman_klass_vol_63d,garch_cond_vol,skew_21d,kurtosis_21d,skew_63d,kurtosis_63d,corwin_schultz_spread,volume_normalized,implied_vol_us,implied_vol_br
2024-12-20,-0.000543,-0.040453,0.095093,0.253222,0.254300,0.283446,0.256546,0.017732,0.217828,0.822124,0.691769,1.765088,0.000000,1.174637,18.360001,16.3193
2024-12-23,-0.026419,-0.063213,0.062718,0.274834,0.260240,0.289060,0.258512,0.017415,0.150789,0.303108,0.651574,1.548142,0.000000,0.990221,16.780001,15.7836
2024-12-26,-0.011781,-0.080216,0.050343,0.279496,0.260275,0.290656,0.259204,0.018089,0.221918,0.115748,0.711044,1.615234,0.002058,0.650005,14.730000,15.4928
2024-12-27,-0.002260,-0.053948,0.043938,0.279871,0.260311,0.293029,0.260298,0.017960,0.277328,0.117679,0.713988,1.615817,0.007221,0.928257,15.950000,15.2315
2024-12-30,-0.010233,-0.051236,0.024878,0.281874,0.257276,0.287882,0.261281,0.017644,0.419597,0.113675,0.792324,1.889242,0.000000,0.697342,17.400000,14.8137


In [5]:
features_panel = pd.concat(features_by_asset, axis=1)
features_panel.to_parquet(PROCESSED_DIR / "features.parquet")
print("features.parquet salvo:", features_panel.shape)

features.parquet salvo: (4336, 111)


## Retorno agregado da carteira (equal-weighted, preparação para a Semana 4/HMM)

Média simples dos `return_1d` dos 7 ativos (já todos em base BRL) — aproximação
padrão para carteira equal-weighted em log-retorno diário. A vol. realizada
do retorno agregado (não a média das vols individuais — isso ignoraria
diversificação) é a segunda dimensão do HMM combinado na Semana 4.

In [6]:
from src.data.features import rolling_realized_volatility

asset_returns = pd.concat({t: df["return_1d"] for t, df in features_by_asset.items()}, axis=1)
portfolio_return_1d = equal_weighted_portfolio_return(asset_returns)
portfolio_vol = rolling_realized_volatility(portfolio_return_1d, windows=tuple(config["features"]["rolling_windows"]))
portfolio_series = pd.concat([portfolio_return_1d, portfolio_vol], axis=1)
portfolio_series.to_parquet(PROCESSED_DIR / "portfolio_return.parquet")
pd.DataFrame({"n_available": asset_returns.notna().sum(axis=1),
              "n_required": asset_returns.shape[1],
              "portfolio_valid": portfolio_return_1d.notna()}).to_csv(TABLES_DIR / "portfolio_coverage.csv")
features_panel.isna().mean().rename("missing_fraction").to_csv(TABLES_DIR / "feature_missingness.csv")
portfolio_series.tail()


,return_1d,realized_vol_21d,realized_vol_63d
2024-12-20,-0.002995,0.180696,0.143490
2024-12-23,-0.016488,0.180925,0.146863
2024-12-26,0.006471,0.181341,0.146824
2024-12-27,-0.011832,0.188284,0.148794
2024-12-30,0.003306,0.185349,0.148803


## Passada 2 — diagnóstico exploratório (ADF/KPSS por ativo; correlação/VIF pooled)

In [7]:
stationarity_by_asset = pd.concat(
    {ticker: stationarity_report(df) for ticker, df in features_by_asset.items()},
    names=["asset", "feature"],
)

# correlação/VIF pooled (todos os ativos empilhados) SÓ fazem sentido aqui porque não
# dependem de ordenação temporal (ao contrário do ADF/KPSS, que exigem uma série por vez)
pooled = pd.concat(features_by_asset.values(), axis=0)
pooled_complete = pooled.dropna()
print(
    "painel pooled (correlação/VIF, diagnóstico exploratório na amostra completa):",
    pooled_complete.shape, "de", pooled.shape[0], "linhas totais antes do dropna",
)

corr = correlation_matrix(pooled_complete)
kept_features, vif_report = vif_filter(pooled_complete, threshold=config["features"]["vif_threshold"])

stationarity_by_asset

painel pooled (correlação/VIF, diagnóstico exploratório na amostra completa): (13818, 16) de 30352 linhas totais antes do dropna


adf_stat    adf_pvalue  \
asset    feature                                          
SLCE3.SA return_1d             -48.533710  0.000000e+00   
         return_5d             -10.691201  3.722087e-19   
         return_21d             -8.757451  2.730945e-14   
         realized_vol_21d       -4.778749  5.971198e-05   
         realized_vol_63d       -4.360085  3.490207e-04   
...                                   ...           ...   
KC=F     kurtosis_63d           -7.439215  6.068214e-11   
         corwin_schultz_spread -16.495829  2.171165e-29   
         volume_normalized     -16.098540  5.161414e-29   
         implied_vol_us         -5.187907  9.305908e-06   
         implied_vol_br         -5.135534  1.188691e-05   

                                adf_rejects_unit_root  kpss_stat  kpss_pvalue  \
asset    feature                                                                
SLCE3.SA return_1d                               True   0.092253     0.100000   
         return_5d                               True   0.106352     0.100000   
         return_21d                              True   0.128745     0.100000   
         realized_vol_21d                        True   1.473629     0.010000   
         realized_vol_63d                        True   1.652055     0.010000   
...                                               ...        ...          ...   
KC=F     kurtosis_63d                            True   0.402808     0.075945   
         corwin_schultz_spread                   True   0.167228     0.100000   
         volume_normalized                       True   0.008660     0.100000   
         implied_vol_us                          True   1.377865     0.010000   
         implied_vol_br                          True   0.936219     0.010000   

                                kpss_fails_to_reject_stationary  \
asset    feature                                                  
SLCE3.SA return_1d                                         True   
         return_5d                                         True   
         return_21d                                        True   
         realized_vol_21d                                 False   
         realized_vol_63d                                 False   
...                                                         ...   
KC=F     kurtosis_63d                                      True   
         corwin_schultz_spread                             True   
         volume_normalized                                 True   
         implied_vol_us                                   False   
         implied_vol_br                                   False   

                                stationary_both_tests  
asset    feature                                       
SLCE3.SA return_1d                               True  
         return_5d                               True  
         return_21d                              True  
         realized_vol_21d                       False  
         realized_vol_63d                       False  
...                                               ...  
KC=F     kurtosis_63d                            True  
         corwin_schultz_spread                   True  
         volume_normalized                       True  
         implied_vol_us                         False  
         implied_vol_br                         False  

[111 rows x 7 columns]

In [8]:
vif_report

,status,vif
feature,,
garman_klass_vol_63d,dropped,10.834360
return_1d,kept,1.255115
return_5d,kept,1.561976
return_21d,kept,1.379772
realized_vol_21d,kept,7.600346
realized_vol_63d,kept,3.276223
garman_klass_vol_21d,kept,3.222212
garch_cond_vol,kept,7.485773
skew_21d,kept,1.418890


In [9]:
stationarity_by_asset.to_csv(TABLES_DIR / "feature_stationarity.csv")
corr.to_csv(TABLES_DIR / "feature_correlation.csv")
vif_report.to_csv(TABLES_DIR / "feature_vif_selection.csv")

print(f"Features retidas após VIF (<= {config['features']['vif_threshold']}) — diagnóstico exploratório, não a seleção final:")
print(kept_features)
print()
print("Features descartadas por multicolinearidade:")
print(vif_report.index[vif_report["status"] == "dropped"].tolist())

Features retidas após VIF (<= 10) — diagnóstico exploratório, não a seleção final:
['return_1d', 'return_5d', 'return_21d', 'realized_vol_21d', 'realized_vol_63d', 'garman_klass_vol_21d', 'garch_cond_vol', 'skew_21d', 'kurtosis_21d', 'skew_63d', 'kurtosis_63d', 'corwin_schultz_spread', 'volume_normalized', 'implied_vol_us', 'implied_vol_br']

Features descartadas por multicolinearidade:
['garman_klass_vol_63d']
